In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_1374.silver.customers_scd2 (
    customer_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    tier STRING,
    signup_date DATE,
    effective_from DATE,
    effective_to DATE,
    is_current BOOLEAN
);

### Change rows classify

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_scd2_changes AS
SELECT
    s.customer_id,
    TRIM(s.full_name) AS full_name,
    LOWER(TRIM(s.email)) AS email,
    REGEXP_REPLACE(s.phone, '[^0-9]', '') AS phone,
    s.city,
    UPPER(TRIM(s.tier)) AS tier,

    CASE
        WHEN t.customer_id IS NULL
            THEN 'NEW_CUSTOMER'

        WHEN NOT (t.city <=> s.city)
          OR NOT (t.tier <=> UPPER(TRIM(s.tier)))
            THEN 'NEW_VERSION'

        WHEN NOT (t.full_name <=> TRIM(s.full_name))
          OR NOT (t.email <=> LOWER(TRIM(s.email)))
          OR NOT (
              t.phone <=> REGEXP_REPLACE(s.phone, '[^0-9]', '')
          )
            THEN 'UPDATE_ONLY'

        ELSE 'NO_CHANGE'
    END AS action_type,

    CAST(s.signup_date AS DATE) AS signup_date

FROM fk_1374.silver.customers s

LEFT JOIN fk_1374.silver.customers_scd2 t
    ON s.customer_id = t.customer_id
   AND t.is_current = TRUE;

In [0]:
%sql
MERGE INTO fk_1374.silver.customers_scd2 AS t

USING (

    -- 1. Existing version ko close karne ke liye
    SELECT
        customer_id AS merge_key,
        customer_id,
        full_name,
        email,
        phone,
        city,
        tier,
        signup_date,
        'CLOSE' AS action_type
    FROM customer_scd2_changes
    WHERE action_type = 'NEW_VERSION'

    UNION ALL

    -- 2. Changed customer ka new version insert karne ke liye
    SELECT
        CAST(NULL AS STRING) AS merge_key,
        customer_id,
        full_name,
        email,
        phone,
        city,
        tier,
        signup_date,
        'INSERT_VERSION' AS action_type
    FROM customer_scd2_changes
    WHERE action_type = 'NEW_VERSION'

    UNION ALL

    -- 3. Sirf name/email/phone update karne ke liye
    SELECT
        customer_id AS merge_key,
        customer_id,
        full_name,
        email,
        phone,
        city,
        tier,
        signup_date,
        'UPDATE_ONLY' AS action_type
    FROM customer_scd2_changes
    WHERE action_type = 'UPDATE_ONLY'

    UNION ALL

    -- 4. Bilkul naye customers
    SELECT
        CAST(NULL AS STRING) AS merge_key,
        customer_id,
        full_name,
        email,
        phone,
        city,
        tier,
        signup_date,
        'NEW_CUSTOMER' AS action_type
    FROM customer_scd2_changes
    WHERE action_type = 'NEW_CUSTOMER'

) AS s

ON t.customer_id = s.merge_key
AND t.is_current = TRUE

-- Purana version close karo
WHEN MATCHED AND s.action_type = 'CLOSE'
THEN UPDATE SET
    t.effective_to = CURRENT_DATE(),
    t.is_current = FALSE

-- Name/email/phone change: existing current version update karo
WHEN MATCHED AND s.action_type = 'UPDATE_ONLY'
THEN UPDATE SET
    t.full_name = s.full_name,
    t.email = s.email,
    t.phone = s.phone

-- New customer ya new historical version insert karo
WHEN NOT MATCHED
 AND s.action_type IN ('INSERT_VERSION', 'NEW_CUSTOMER')
THEN INSERT (
    customer_id,
    full_name,
    email,
    phone,
    city,
    tier,
    signup_date,
    effective_from,
    effective_to,
    is_current
)
VALUES (
    s.customer_id,
    s.full_name,
    s.email,
    s.phone,
    s.city,
    s.tier,
    s.signup_date,
    CASE
        WHEN s.action_type = 'NEW_CUSTOMER'
            THEN s.signup_date
        ELSE CURRENT_DATE()
    END,
    NULL,
    TRUE
);